# Stage 6b — Prior Admission ICD Carry-Forward

**Input** : `patient_records/<patient>/admission_history.json`  
**Output**: `stage_06b_history_context/history_codes.json` per admission

## What this stage does
For each admission, collect all ICD-10 codes from the **same patient's prior admissions**
using the `admission_history.json` file (written by Stage 4), and carry them forward as
candidate codes for the current admission.

These codes are already verified — no UMLS lookup, no grounding, no scoring needed.
Chronic conditions (diabetes, cirrhosis, CKD, etc.) tend to persist across admissions,
so prior codes are a strong signal for what will appear on the current discharge summary.

Codes seen across multiple prior admissions are ranked higher (more likely to be chronic).

## Pipeline position
```
Stage 5: Evidence Scoring + Pruning        ✅
Stage 6: SNOMED Grounding (active)         ✅
Stage 6b: Prior Admission Carry-Forward    ← THIS NOTEBOOK
Stage 7:  Final ICD Decision               (next)
```

## 1. Setup

In [1]:
import json
from pathlib import Path
import pandas as pd

RECORDS_DIR    = Path(r'C:\Users\esnam\OneDrive\Desktop\esna_master_proj\ai-agents-for-clinical-coding\patient_records')
STAGE6_OUTPUT  = 'stage_06_snomed_grounding'
STAGE6B_OUTPUT = 'stage_06b_history_context'

patients = sorted([p for p in RECORDS_DIR.iterdir() if p.is_dir() and p.name.startswith('patient_')])
print(f'Patients found: {len(patients)}')

Patients found: 15


## 2. Load Prior Codes from admission_history.json

In [2]:
def load_admission_history(patient_dir: Path) -> list[dict]:
    """Load admission_history.json from the patient-level folder.
    
    Returns a list of prior admission dicts, each with:
      - admission_id, admittime, dischtime, admission_type
      - primary_icd_code, primary_dx_title
      - ground_truth_icd10: list of ICD-10 codes (no dots)
      - ground_truth_dx_titles: parallel list of descriptions
    
    Returns [] if the file doesn't exist.
    """
    hist_path = patient_dir / 'admission_history.json'
    if not hist_path.exists():
        return []
    with open(hist_path, encoding='utf-8') as f:
        return json.load(f)


print('admission_history.json loader defined.')

admission_history.json loader defined.


## 3. Collect Prior Admission Codes for All Patients

In [ ]:
all_history = []

for patient_dir in patients:
    patient_id = patient_dir.name.replace('patient_', '')
    adm_root   = patient_dir / 'admissions'
    adm_dirs   = sorted(adm_root.iterdir()) if adm_root.exists() else []

    prior_admissions = load_admission_history(patient_dir)

    # Collect which codes were ever the primary diagnosis in a prior admission
    primary_codes_set = {
        str(adm.get('primary_icd_code', '')).upper()
        for adm in prior_admissions
        if adm.get('primary_icd_code')
    }

    for adm_dir in adm_dirs:
        prior_codes: dict[str, dict] = {}
        for adm in prior_admissions:
            src_id = str(adm.get('admission_id', adm.get('hadm_id', '')))
            codes  = adm.get('ground_truth_icd10', [])
            titles = adm.get('ground_truth_dx_titles', [])
            primary = str(adm.get('primary_icd_code', '')).upper()
            for code, title in zip(codes, titles + [''] * len(codes)):
                code = str(code).upper()
                if code not in prior_codes:
                    prior_codes[code] = {
                        'icdCode'          : code,
                        'title'            : title,
                        'source_admissions': [],
                        'primary_count'    : 0,   # times this was the primary dx
                        'was_primary'      : False,
                    }
                prior_codes[code]['source_admissions'].append(src_id)
                if code == primary:
                    prior_codes[code]['primary_count'] += 1
                    prior_codes[code]['was_primary'] = True

        # Sort: primary codes first, then by frequency
        prior_list = sorted(
            prior_codes.values(),
            key=lambda x: (x['primary_count'], len(x['source_admissions'])),
            reverse=True,
        )

        output = {
            'patient_id'        : patient_id,
            'admission_id'      : adm_dir.name.replace('hadm_', ''),
            'n_prior_admissions': len(prior_admissions),
            'n_prior_codes'     : len(prior_list),
            'prior_icd_codes'   : prior_list,
        }

        out_dir = adm_dir / STAGE6B_OUTPUT
        out_dir.mkdir(exist_ok=True)
        with open(out_dir / 'history_codes.json', 'w') as f:
            json.dump(output, f, indent=2)

        all_history.append(output)
        n_primary = sum(1 for c in prior_list if c['was_primary'])
        print(f'Patient {patient_id} | {adm_dir.name} | '
              f'prior_admissions={len(prior_admissions)} '
              f'prior_codes={len(prior_list)} primary_codes={n_primary}')

print(f'\nDone. {len(all_history)} admissions processed.')

## 4. Inspect One Admission

In [4]:
EXAMPLE_IDX = 5  # patient_11607177
ex = all_history[EXAMPLE_IDX]

print(f'Patient          : {ex["patient_id"]}')
print(f'Admission        : {ex["admission_id"]}')
print(f'Prior admissions : {ex["n_prior_admissions"]}')
print(f'Prior ICD codes  : {ex["n_prior_codes"]}')
print()

if ex['prior_icd_codes']:
    print(f'{"Code":<12} {"Seen":>5}  {"Description"}')
    print('-' * 70)
    for c in ex['prior_icd_codes'][:25]:
        freq = len(c['source_admissions'])
        print(f'  {c["icdCode"]:<10}  x{freq}  {c["title"][:55]}')
else:
    print('  No prior admissions found in admission_history.json.')

Patient          : 11607177
Admission        : 23293838
Prior admissions : 4
Prior ICD codes  : 39

Code          Seen  Description
----------------------------------------------------------------------
  I420        x4  Dilated cardiomyopathy
  E119        x4  Type 2 diabetes mellitus without complications
  E669        x4  Obesity, unspecified
  Z8673       x4  Personal history of transient ischemic attack (TIA), an
  I5022       x3  Chronic systolic (congestive) heart failure
  I472        x3  Ventricular tachycardia
  N179        x3  Acute kidney failure, unspecified
  I10         x3  Essential (primary) hypertension
  I272        x3  Other secondary pulmonary hypertension
  M109        x3  Gout, unspecified
  E785        x3  Hyperlipidemia, unspecified
  Z95810      x3  Presence of automatic (implantable) cardiac defibrillat
  Z7682       x3  Awaiting organ transplant status
  Z7901       x3  Long term (current) use of anticoagulants
  Z87891      x3  Personal history of nicotine 

## 5. Evaluate — Stage 6 + Prior Codes Combined

In [5]:
def parse_ground_truth(gt_path):
    """Parse ground_truth.txt → list of icdCode strings."""
    if not gt_path.exists():
        return []
    codes = []
    with open(gt_path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line and line[0].isdigit() and '—' in line:
                parts    = line.split('—', 1)
                raw_code = parts[0].strip()
                code     = raw_code.split('.')[-1].strip().split()[0].upper()
                if code:
                    codes.append(code)
    return codes


def load_ground_truth(patient_dir, hadm_id):
    gt_path = patient_dir / 'admissions' / f'hadm_{hadm_id}' / 'ground_truth.txt'
    return parse_ground_truth(gt_path)


def _prefix_match(pred, gt_set):
    if pred in gt_set:
        return True
    for gt in gt_set:
        if gt.startswith(pred) or pred.startswith(gt):
            return True
    return False


def _f1(tp, pred, gt):
    p = len(tp) / len(pred) if pred else 0.0
    r = len(tp) / len(gt)   if gt   else 0.0
    return round(2*p*r/(p+r) if (p+r) > 0 else 0.0, 3)


all_metrics = []

for hist in all_history:
    pid  = hist['patient_id']
    hadm = hist['admission_id']
    pdir = RECORDS_DIR / f'patient_{pid}'
    gt   = load_ground_truth(pdir, hadm)
    if not gt:
        continue
    gt_set = set(gt)

    # Stage 6 codes (from SNOMED grounding)
    s6_path  = pdir / 'admissions' / f'hadm_{hadm}' / STAGE6_OUTPUT / 'grounded_symptoms.json'
    s6_codes = set()
    if s6_path.exists():
        with open(s6_path) as f:
            s6 = json.load(f)
        for b in s6.get('grounded_branches', []):
            for s in b.get('grounded_symptoms', []):
                for c in s.get('icd_candidates', []):
                    s6_codes.add(c['icdCode'].replace('.', '').upper())

    # Stage 6b — prior admission codes from admission_history.json
    s6b_codes = {e['icdCode'] for e in hist['prior_icd_codes']}

    combined = s6_codes | s6b_codes

    tp_s6   = {p for p in s6_codes  if _prefix_match(p, gt_set)}
    tp_s6b  = {p for p in s6b_codes if _prefix_match(p, gt_set)}
    tp_comb = {p for p in combined  if _prefix_match(p, gt_set)}

    all_metrics.append({
        'patient_id'        : pid,
        'n_prior_admissions': hist['n_prior_admissions'],
        'n_gt'              : len(gt_set),
        'n_s6'              : len(s6_codes),
        'f1_s6'             : _f1(tp_s6,   s6_codes,  gt_set),
        'n_s6b'             : len(s6b_codes),
        'f1_s6b'            : _f1(tp_s6b,  s6b_codes, gt_set),
        'n_combined'        : len(combined),
        'f1_combined'       : _f1(tp_comb, combined,   gt_set),
    })

df   = pd.DataFrame(all_metrics)
cols = ['patient_id', 'n_prior_admissions', 'n_gt', 'n_s6', 'f1_s6', 'n_s6b', 'f1_s6b', 'n_combined', 'f1_combined']
print(df[cols].to_string(index=False))
print(f'\nMean F1 — Stage 6 only       : {df["f1_s6"].mean():.3f}')
print(f'Mean F1 — Prior codes only   : {df["f1_s6b"].mean():.3f}')
print(f'Mean F1 — Combined 6 + prior : {df["f1_combined"].mean():.3f}')

patient_id  n_prior_admissions  n_gt  n_s6  f1_s6  n_s6b  f1_s6b  n_combined  f1_combined
  10361982                   1     5     0  0.000     11   0.250          11        0.250
  10426859                   2    22    10  0.000     24   0.609          34        0.500
  10458324                   1    12    30  0.238      8   0.100          38        0.240
  11251337                   2     7    12  0.000     12   0.211          24        0.129
  11474876                   1    17     0  0.000     23   0.450          23        0.450
  11607177                   4    13     0  0.000     39   0.462          39        0.462
  12007928                   2    19     5  0.000     31   0.560          36        0.509
  13196707                   1    32    11  0.047     33   0.246          44        0.237
  13508515                   1    14     5  0.000     16   0.333          21        0.286
  13952483                   2    25    20  0.178     46   0.394          65        0.400
  16014068

## 6. Save Summary

In [6]:
summary = {
    'stage'             : 'stage_06b_prior_admission_carry_forward',
    'n_admissions'      : len(all_history),
    'mean_f1_s6'        : round(df['f1_s6'].mean(), 3),
    'mean_f1_s6b'       : round(df['f1_s6b'].mean(), 3),
    'mean_f1_combined'  : round(df['f1_combined'].mean(), 3),
    'stage5_baseline_f1': 0.028,
    'per_patient'       : all_metrics,
}

out_path = RECORDS_DIR / 'stage_06b_summary.json'
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(f'Saved: {out_path}')
print(f'\nStage 5 baseline F1       : {summary["stage5_baseline_f1"]}')
print(f'Stage 6 mean F1           : {summary["mean_f1_s6"]}')
print(f'Prior codes mean F1       : {summary["mean_f1_s6b"]}')
print(f'Combined 6 + prior F1     : {summary["mean_f1_combined"]}')

Saved: C:\Users\esnam\OneDrive\Desktop\esna_master_proj\ai-agents-for-clinical-coding\patient_records\stage_06b_summary.json

Stage 5 baseline F1       : 0.028
Stage 6 mean F1           : 0.106
Prior codes mean F1       : 0.363
Combined 6 + prior F1     : 0.373
